![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 05: Knowledge Agents and Stateful Workflows)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 5A: Basic RAG System

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local RAG system using approved public snippets and lexical retrieval</td></tr>
<tr><td align="left">Optional part</td><td>Embedding retrieval or real model answer generation if packages and API key are available</td></tr>
<tr><td align="left">Main output</td><td>A small RAG pipeline with retrieval, grounded answer construction, source display, limitations and tests</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m05a-overview)
2. [Conceptual Background: What RAG Solves](#m05a-background)
3. [Setup](#m05a-setup)
4. [Approved Knowledge Base](#m05a-knowledge-base)
5. [Mandatory Local RAG Pipeline](#m05a-local-rag)
6. [Inspecting Retrieval and Grounding](#m05a-inspection)
7. [Optional Embeddings or Real Model Section](#m05a-optional)
8. [Testing and Analysis](#m05a-testing)
9. [Student Tasks](#m05a-student-tasks)
10. [Submission and Reflection](#m05a-submission)

---

<a id="m05a-overview"></a>

### 1. Overview and Learning Goals

M05 starts the knowledge-agent part of the unit. In M02C, you studied embeddings and vector similarity. In M03C, you built a visual RAG workflow in Flowise. In M04, you implemented prompt, parser, tool and controlled-action patterns in Python. M05A now builds a small RAG system directly in code.

RAG means **retrieval-augmented generation**. The system does not answer only from the model's internal knowledge. It first retrieves relevant context from an approved knowledge base, then constructs an answer using that retrieved context.

```mermaid
flowchart LR
    A[User question] --> B[Retriever]
    C[Approved knowledge base] --> B
    B --> D[Retrieved context]
    D --> E[Grounded answer builder]
    E --> F[Answer with sources and limitations]
```

This notebook uses a mandatory local RAG pipeline. It does not require an API key. It uses approved public snippets and a simple lexical retriever. The purpose is to make the RAG architecture visible and testable before using embeddings, vector databases or real model calls.

By the end of this session, you should be able to:

```text
1. Explain the difference between retrieval and generation.
2. Build a small local retriever.
3. Construct an answer only from retrieved context.
4. Display sources with the answer.
5. Detect insufficient context.
6. Test normal, weak-evidence and invalid-input cases.
7. Explain why RAG systems need retrieval inspection and source grounding.
```

<a id="m05a-background"></a>

### 2. Conceptual Background: What RAG Solves

#### 2.1 The problem with answering from the model alone

A language model can often produce a fluent answer even when it does not have the right evidence. This is useful for general writing, but risky for knowledge-intensive systems. In a unit assistant, a research assistant, a policy assistant or a technical support assistant, the answer should be based on specific approved information.

A model-only workflow is:

```mermaid
flowchart LR
    A[Question] --> B[Prompt]
    B --> C[Model]
    C --> D[Answer]
```

The weakness is that the answer may not be grounded in the material you want the system to use. It may sound plausible while inventing details.

A RAG workflow adds an evidence step:

```mermaid
flowchart LR
    A[Question] --> B[Retrieve relevant context]
    C[Knowledge base] --> B
    B --> D[Prompt with context]
    D --> E[Model or answer builder]
    E --> F[Grounded answer]
```

The model is no longer asked to answer from nowhere. It is asked to answer from retrieved evidence.

#### 2.2 Retrieval is not generation

Retrieval and generation are different jobs.

<div align="center">

<table>
<thead>
<tr><th><strong>Stage</strong></th><th><strong>Question it answers</strong></th><th><strong>Typical failure</strong></th><th><strong>What to inspect</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Retrieval</td><td>Which documents are relevant?</td><td>Wrong or missing context.</td><td>Retrieved documents, scores, source titles.</td></tr>
<tr><td align="left">Generation</td><td>How should the answer be written?</td><td>Unsupported or over-confident answer.</td><td>Whether the answer is supported by retrieved context.</td></tr>
</tbody>
</table>

</div>

A RAG system can fail even if the final answer sounds good. For example, the retriever may return the wrong document, and the answer builder may still write a confident answer. Therefore, in this notebook, every answer will display the retrieved sources and limitations.

#### 2.3 What does “grounded” mean?

An answer is grounded when its claims can be traced back to retrieved context. If the retrieved documents do not say something, the answer should not claim it. If the retrieved context is weak or missing, the system should say that the available context is insufficient.

Good RAG behaviour:

```text
Question: What does M05A teach?
Retrieved source: M05A snippet about retrieval and grounding.
Answer: M05A teaches local RAG, retrieval, sources and insufficient-context handling.
```

Bad RAG behaviour:

```text
Question: What is the final exam room?
Retrieved source: no relevant document.
Answer: The final exam is in Room 201.
```

The second answer is unacceptable because it invents information not contained in the approved context.

#### 2.4 How this connects to previous modules

<div align="center">

<table>
<thead>
<tr><th><strong>Previous session</strong></th><th><strong>Concept carried into M05A</strong></th><th><strong>How it appears here</strong></tr>
</thead>
<tbody>
<tr><td align="left">M02C</td><td>Similarity and representation.</td><td>We begin with lexical similarity and later compare this with embeddings.</td></tr>
<tr><td align="left">M03C</td><td>Visual RAG in Flowise.</td><td>The same RAG pipeline is now implemented in code.</td></tr>
<tr><td align="left">M04A</td><td>Prompt/model/parser pipeline.</td><td>The answer builder plays the role of a controlled generation component.</td></tr>
<tr><td align="left">M04D</td><td>Evidence-grounded drafting.</td><td>Answers must show evidence and limitations.</td></tr>
</tbody>
</table>

</div>

<a id="m05a-setup"></a>

### 3. Setup

This notebook uses only standard Python for the mandatory section. It does not require a real API key. The optional section includes examples for a real model call, but the required learning outcome is the local RAG pipeline.

The mandatory pipeline is:

```text
approved documents -> lexical retriever -> retrieved context -> conservative answer builder -> answer with sources
```

Use only approved public or synthetic teaching material. Do not index private documents, student submissions, hidden instructor solutions, credentials, emails or personal records.

In [ ]:
import json
import re
from typing import Any, Dict, List

print("M05A setup complete.")

<a id="m05a-knowledge-base"></a>

### 4. Approved Knowledge Base

The knowledge base below simulates public unit material. Each document is short so that you can inspect retrieval behaviour manually.

Each document has:

```text
doc_id: a stable identifier
title: short title
text: approved content
source: source description
tags: topic labels used to improve retrieval
```

A real RAG system may use PDFs, web pages, Markdown files, notebooks or databases. However, the same principle applies: only approved documents should be indexed.

In [ ]:
KNOWLEDGE_BASE = [
    {
        "doc_id": "D001",
        "title": "Flowise and Visual Workflows",
        "text": "Flowise represents AI workflows visually. Students connect nodes such as prompts, chat models, retrievers, tools and outputs to understand information flow.",
        "source": "public unit snippet",
        "tags": ["flowise", "visual_workflow", "nodes", "tools"]
    },
    {
        "doc_id": "D002",
        "title": "LangChain Code Workflows",
        "text": "LangChain-style workflows express prompt templates, model calls, parsers and chains in Python code. This makes AI workflows testable and reusable.",
        "source": "public unit snippet",
        "tags": ["langchain", "prompt", "parser", "chain"]
    },
    {
        "doc_id": "D003",
        "title": "RAG Fundamentals",
        "text": "RAG retrieves relevant context before constructing an answer. A good RAG system should show sources and avoid answering beyond the retrieved evidence.",
        "source": "public unit snippet",
        "tags": ["rag", "retrieval", "sources", "grounding"]
    },
    {
        "doc_id": "D004",
        "title": "Tool Agent Safety",
        "text": "Tool agents can call approved functions, but they need validation and refusal rules. Unsafe tools such as shell commands or private-file access should be excluded from early labs.",
        "source": "public unit snippet",
        "tags": ["tool_agents", "safety", "validation", "refusal"]
    },
    {
        "doc_id": "D005",
        "title": "Stateful Workflows",
        "text": "Stateful workflows keep track of decisions, intermediate outputs and transitions. LangGraph can represent branching workflows such as success, validation error and refusal.",
        "source": "public unit snippet",
        "tags": ["langgraph", "state", "workflow", "branching"]
    },
]

print("Number of documents:", len(KNOWLEDGE_BASE))
print(json.dumps(KNOWLEDGE_BASE[0], indent=2))

#### 4.1 Why these documents are small

Small snippets are easier to inspect. When you run a query, you can quickly see whether the right document was retrieved. This is valuable for learning because RAG quality depends strongly on retrieval quality.

In a production RAG system, documents are often split into chunks. Each chunk should be large enough to preserve meaning but small enough to retrieve precisely. This notebook does not implement full chunking yet; M05B will move closer to unit-material RAG.

<a id="m05a-local-rag"></a>

### 5. Mandatory Local RAG Pipeline

#### 5.1 Tokenisation and lexical scoring

The local retriever needs a way to compare the question with each document. We start with a simple method: split text into terms and count overlap between query terms and document terms.

This is called lexical retrieval because it depends on words. It is not semantic retrieval. If the question uses very different words from the document, lexical retrieval may miss relevant material. This limitation is useful to observe before using embeddings.

In [ ]:
def tokenize(text: str) -> List[str]:
    """Split text into lowercase word-like tokens."""
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z_]+", text.lower())


print(tokenize("RAG retrieves relevant context before answering."))

In [ ]:
def score_document(query: str, document: Dict[str, Any]) -> int:
    """Score a document by keyword overlap with the query."""

    query_terms = set(tokenize(query))
    doc_text = " ".join([
        document.get("title", ""),
        document.get("text", ""),
        " ".join(document.get("tags", [])),
    ])
    doc_terms = set(tokenize(doc_text))

    return len(query_terms.intersection(doc_terms))


for doc in KNOWLEDGE_BASE:
    print(doc["doc_id"], doc["title"], "score=", score_document("How does RAG use retrieved sources?", doc))

The score is simple: more overlapping terms means a higher score. This is not the only retrieval method, but it makes retrieval transparent. You should be able to explain why a document was retrieved.

In [ ]:
def retrieve_documents(query: str, knowledge_base: List[Dict[str, Any]], top_k: int = 3) -> Dict[str, Any]:
    """Retrieve top-k documents using lexical overlap."""

    if not isinstance(query, str) or not query.strip():
        return {"ok": False, "error": "query must be a non-empty string.", "result": None}

    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    scored = []
    for doc in knowledge_base:
        score = score_document(query, doc)
        if score > 0:
            scored.append((score, doc))

    scored.sort(key=lambda pair: pair[0], reverse=True)

    results = []
    for score, doc in scored[:top_k]:
        item = dict(doc)
        item["score"] = score
        results.append(item)

    return {"ok": True, "error": None, "result": results}


retrieved = retrieve_documents("How does RAG use retrieved sources?", KNOWLEDGE_BASE)
retrieved

#### 5.2 Grounded answer construction

The answer builder below is deliberately conservative. It does not invent new facts. It either builds an answer from retrieved snippets or says that the available context is insufficient.

In a real RAG system, this step is often performed by a language model. Even then, the instruction should be similar:

```text
Use only the retrieved context.
If the context is insufficient, say so.
Show the sources.
Do not invent details.
```

In [ ]:
def build_grounded_answer(question: str, retrieved_docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Build a conservative answer using only retrieved documents."""

    if not retrieved_docs:
        return {
            "ok": True,
            "error": None,
            "result": {
                "answer": "The available approved context does not contain enough information to answer this question.",
                "sources": [],
                "limitations": ["No relevant approved document was retrieved."],
            },
        }

    combined_context = " ".join(doc["text"] for doc in retrieved_docs)

    answer = (
        "Based on the retrieved approved context, "
        f"{combined_context}"
    )

    return {
        "ok": True,
        "error": None,
        "result": {
            "answer": answer,
            "sources": [
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "source": doc["source"],
                    "score": doc["score"],
                }
                for doc in retrieved_docs
            ],
            "limitations": [
                "The answer is generated only from the retrieved approved snippets.",
                "If the retrieved context is weak or incomplete, the answer may be incomplete.",
            ],
        },
    }


answer = build_grounded_answer("How does RAG use retrieved sources?", retrieved["result"])
answer

#### 5.3 Full local RAG system

Now we combine retrieval and answer construction into one class. This mirrors the structure of M04 chain design, but adds a retrieval stage.

In [ ]:
class LocalRAGSystem:
    """A transparent local RAG system for teaching retrieval and grounding."""

    def __init__(self, knowledge_base: List[Dict[str, Any]]):
        self.knowledge_base = knowledge_base

    def invoke(self, question: str, top_k: int = 3) -> Dict[str, Any]:
        retrieval = retrieve_documents(question, self.knowledge_base, top_k=top_k)
        if not retrieval["ok"]:
            return retrieval

        answer = build_grounded_answer(question, retrieval["result"])
        if not answer["ok"]:
            return answer

        return {
            "ok": True,
            "error": None,
            "result": {
                "question": question,
                "retrieved_docs": retrieval["result"],
                "answer": answer["result"],
            },
        }


rag = LocalRAGSystem(KNOWLEDGE_BASE)
rag_result = rag.invoke("How does RAG use retrieved sources?")
rag_result

<a id="m05a-inspection"></a>

### 6. Inspecting Retrieval and Grounding

A RAG system should not be evaluated only by reading the final answer. You should inspect:

```text
1. the question,
2. the retrieved documents,
3. the retrieval scores,
4. the answer,
5. the sources,
6. the limitations.
```

The display function below makes those parts visible.

In [ ]:
def display_rag_result(rag_result: Dict[str, Any]) -> None:
    if not rag_result.get("ok"):
        print("ERROR:", rag_result.get("error"))
        return

    result = rag_result["result"]
    print("Question:", result["question"])

    print("\nRetrieved documents:")
    if not result["retrieved_docs"]:
        print("- None")
    for doc in result["retrieved_docs"]:
        print(f"- {doc['doc_id']} | score={doc['score']} | {doc['title']}")
        print(f"  Text: {doc['text']}")

    print("\nAnswer:")
    print(result["answer"]["answer"])

    print("\nSources:")
    if not result["answer"]["sources"]:
        print("- None")
    for source in result["answer"]["sources"]:
        print(f"- {source['doc_id']} | {source['title']} | {source['source']} | score={source['score']}")

    print("\nLimitations:")
    for limitation in result["answer"]["limitations"]:
        print("-", limitation)


display_rag_result(rag_result)

#### 6.1 Interpreting retrieval results

A retrieval result should be judged before the answer is trusted.

<div align="center">

<table>
<thead>
<tr><th><strong>Retrieval outcome</strong></th><th><strong>Interpretation</strong></th><th><strong>What to do</strong></tr>
</thead>
<tbody>
<tr><td align="left">Relevant document retrieved with high score</td><td>The answer has a useful evidence base.</td><td>Check whether the answer stays within the source.</td></tr>
<tr><td align="left">Only weakly related document retrieved</td><td>The answer may be incomplete or misleading.</td><td>Improve query, document tags or retrieval method.</td></tr>
<tr><td align="left">No document retrieved</td><td>The system lacks approved context.</td><td>Return insufficient-context response.</td></tr>
<tr><td align="left">Wrong document retrieved</td><td>The answer should not be trusted.</td><td>Debug retrieval before changing the answer generator.</td></tr>
</tbody>
</table>

</div>

A common mistake is trying to fix every RAG failure by changing the model. In many cases, the real problem is retrieval: the right context was never provided.

<a id="m05a-optional"></a>

### 7. Optional Embeddings or Real Model Section

This section is optional. The mandatory RAG pipeline already teaches the core architecture. If packages and API access are available, you may later replace lexical retrieval with embeddings or replace the conservative answer builder with a real model call.

The architecture should remain the same:

```mermaid
flowchart LR
    A[Question] --> B[Retriever]
    B --> C[Retrieved context]
    C --> D[Prompt]
    D --> E[Real model]
    E --> F[Answer with sources]
```

Do not hard-code API keys. Do not index private documents. If you do not have a valid API key, write:

```text
Skipped: no API key available.
```

In [ ]:
# Optional package installation.
# Uncomment only if package installation is allowed.

# !pip install -q langchain langchain-core langchain-openai

In [ ]:
import os

has_openai_key = bool(os.environ.get("OPENAI_API_KEY"))
print("OPENAI_API_KEY found:", has_openai_key)

if not has_openai_key:
    print("Skipped optional real-model section: no API key available.")

In [ ]:
def optional_real_rag_answer(question: str, retrieved_docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Optional real-model answer generation from retrieved context."""

    import os

    if not os.environ.get("OPENAI_API_KEY"):
        return {"ok": False, "error": "OPENAI_API_KEY is not set. Skip this optional section.", "result": None}

    try:
        from langchain_openai import ChatOpenAI
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.output_parsers import StrOutputParser
    except ImportError as exc:
        return {"ok": False, "error": f"Required packages are not installed: {exc}", "result": None}

    context = "\n\n".join([f"{doc['doc_id']} {doc['title']}: {doc['text']}" for doc in retrieved_docs])

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using only the provided context. If the context is insufficient, say so. Include source document ids."),
        ("human", "Question: {question}\n\nContext:\n{context}")
    ])

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
    chain = prompt | model | StrOutputParser()

    return {"ok": True, "error": None, "result": chain.invoke({"question": question, "context": context})}


optional_output = optional_real_rag_answer(
    "How does RAG use sources?",
    retrieve_documents("How does RAG use sources?", KNOWLEDGE_BASE)["result"],
)
optional_output

If the optional model call runs, compare it with the conservative answer builder:

```text
Does the real model use only retrieved context?
Does it cite document ids?
Does it add unsupported claims?
Does it behave correctly when context is insufficient?
```

The optional model may produce more fluent text. Fluency is not the same as grounding.

<a id="m05a-testing"></a>

### 8. Testing and Analysis

RAG tests should check both retrieval and answer behaviour. The tests below cover normal retrieval, topic-specific retrieval, weak/no evidence and invalid input.

In [ ]:
test_rag = LocalRAGSystem(KNOWLEDGE_BASE)

# Normal: RAG topic.
rag_case = test_rag.invoke("What is RAG and why should it show sources?")
assert rag_case["ok"] is True
assert len(rag_case["result"]["retrieved_docs"]) >= 1
assert any(doc["doc_id"] == "D003" for doc in rag_case["result"]["retrieved_docs"])
assert len(rag_case["result"]["answer"]["sources"]) >= 1

# Normal: tool agent safety.
tool_case = test_rag.invoke("Why do tool agents need validation and refusal rules?")
assert tool_case["ok"] is True
assert any(doc["doc_id"] == "D004" for doc in tool_case["result"]["retrieved_docs"])

# Normal: LangGraph state.
state_case = test_rag.invoke("How can stateful workflows represent branching?")
assert state_case["ok"] is True
assert any(doc["doc_id"] == "D005" for doc in state_case["result"]["retrieved_docs"])

# Weak/no evidence.
weak_case = test_rag.invoke("What is the policy for final exam room allocation?")
assert weak_case["ok"] is True
assert weak_case["result"]["retrieved_docs"] == []
assert weak_case["result"]["answer"]["sources"] == []
assert "does not contain enough information" in weak_case["result"]["answer"]["answer"]

# Failure: empty question.
empty = test_rag.invoke("")
assert empty["ok"] is False

# Failure: invalid top_k.
bad_top_k = test_rag.invoke("RAG", top_k=0)
assert bad_top_k["ok"] is False

print("All M05A mandatory local-RAG tests passed.")

In [ ]:
for question in [
    "What is RAG and why should it show sources?",
    "Why do tool agents need validation?",
    "What is the final exam room?",
]:
    print("\n==============================")
    display_rag_result(test_rag.invoke(question))

The third example should return an insufficient-context answer. This is correct behaviour. A safe RAG system should not invent exam-room information when no relevant approved document is retrieved.

<a id="m05a-student-tasks"></a>

### 9. Student Tasks

Complete the tasks below. The mandatory local RAG system must run without external API calls.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What to do</strong></th><th><strong>Detailed instructions</strong></th><th><strong>Evidence to submit</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run the provided RAG pipeline.</td><td>Run all cells through the mandatory testing section.</td><td>Output showing <code>All M05A mandatory local-RAG tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add one document</td><td>Add a new approved document.</td><td>Add a document to <code>KNOWLEDGE_BASE</code> with <code>doc_id</code>, <code>title</code>, <code>text</code>, <code>source</code> and <code>tags</code>. It must be public-style or synthetic teaching content. Do not use private data.</td><td>Updated knowledge-base item.</td></tr>
<tr><td align="left">Task 3: Query your document</td><td>Run the RAG system on a matching question.</td><td>Your new document should be retrieved in the top results. Use <code>display_rag_result</code> to show the retrieved document, answer, sources and limitations.</td><td>Displayed retrieved document, answer and sources.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add at least three tests.</td><td>Include one test retrieving your new document, one weak/no-evidence test and one invalid-input test. Use <code>assert</code> statements.</td><td>Test cell with passing assertions.</td></tr>
<tr><td align="left">Task 5: Analyse grounding</td><td>Explain whether the answer is grounded.</td><td>Identify which source was used, which claim came from that source, and whether the answer added unsupported claims.</td><td>Short grounding paragraph.</td></tr>
<tr><td align="left">Task 6: Optional real model</td><td>Run or skip the optional model section.</td><td>If you have API access, run it safely. If not, write <code>Skipped: no API key available</code>.</td><td>Real output or skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Write a short explanation.</td><td>Explain why RAG needs retrieval inspection, source display and insufficient-context handling.</td><td>150–250 words.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter.
# Add one approved public-style document.

# Example:
# new_doc = {
#     "doc_id": "D006",
#     "title": "Human Review in AI Workflows",
#     "text": "Human review is important when AI systems generate drafts, make recommendations or use tools. Review helps detect unsupported claims and unsafe outputs.",
#     "source": "synthetic public teaching snippet",
#     "tags": ["human_review", "safety", "drafting", "tools"]
# }
#
# KNOWLEDGE_BASE.append(new_doc)
# student_rag = LocalRAGSystem(KNOWLEDGE_BASE)
# display_rag_result(student_rag.invoke("Why is human review important for AI drafts?"))

<a id="m05a-submission"></a>

### 10. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your new knowledge-base document.
3. RAG output showing your new document retrieved.
4. At least three added tests using assert statements.
5. Short grounding analysis.
6. Optional real-model result or skipped note.
7. 150–250 word reflection.
```

Reflection questions:

1. What is the difference between retrieval and generation?
2. Why should a RAG answer show sources?
3. What should the system do when no relevant context is retrieved?
4. What is the risk of answering beyond retrieved evidence?
5. How does this prepare for M05B unit-material RAG and M05C LangGraph?

#### Further Readings

- LangChain RAG tutorials: <https://python.langchain.com/docs/tutorials/rag/>
- LangChain retrievers: <https://python.langchain.com/docs/concepts/retrievers/>
- LangChain vector stores: <https://python.langchain.com/docs/concepts/vectorstores/>
- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>